# Polymarket vs Kalshi: activity and last-print gaps by competition

This notebook studies the full population's trading activity, then price gaps, fee-adjusted last-print screens, frequency and episode duration in candidate competition-season × phase cells. A candidate uses the smallest F in 5 / 10 / 60 seconds reaching 70% dual-fresh coverage (§2b). The full-population state split (§2a) runs before this filter; the finer activity diagnostics (§2c–d) run inside it.

**Contents:** §1 population; §2 availability (full population), candidate cells, then per-action-pair availability and directional trade waiting inside the candidates; §3–4 signed price gaps; §5 fee-adjusted last-print screens; §6 maker/maker by direction across F; §7 conditions; §8 frequency. Event-time episodes moved to [02.2 §5](02.2_cross_venue_confirmation_and_response.ipynb). Subsequent price-response, passive-fill and basket research is in [02.2](02.2_cross_venue_confirmation_and_response.ipynb).

**Inputs:** `availability.csv`, `trade_waits.parquet` from [`availability.py`](../availability.py); `gap_cents.csv` from [`gap_cents.py`](../gap_cents.py); `screen_sweep.csv` from [`screen_sweep.py`](../screen_sweep.py); `matches`, `audit`, `screen`, `conditions` from `gaps.py`. The screen and conditions use each candidate's F. Shared candidate selection and chart helpers live in [`competition.py`](../competition.py).

**Vocabulary.** A print aggregates a venue's fills within one second, outcome and taker side into a volume-weighted price; the pooled `all` leg merges both sides. A pair combines the venues' latest prints. Its age is the older print's age; both are fresh when age ≤ F. An observation is a print on either pair leg while the other leg is fresh. A route assigns venue roles and which venue holds YES. F looks backward; the activity waiting horizon H looks forward.

Trade prices are references, not executable quotes. The tape has no books, depth, queues or fills, and these screens do not establish P&L.

In [ ]:
from pathlib import Path
import subprocess, sys
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'research/soccer_1x2_analysis/ticks.py').exists())
ANALYSIS = ROOT / 'research/soccer_1x2_analysis'
sys.path.insert(0, str(ANALYSIS))
import gaps, ticks

OUT = ANALYSIS / 'outputs/gap_analysis'
CFG = gaps.Config()
STRICT = CFG.freshness_sec
PHASES = list(gaps.PHASES)
PHASE_SEC = gaps.phase_seconds(CFG)
def load(name):
    return pd.read_parquet(OUT / name) if name.endswith('.parquet') else pd.read_csv(OUT / name)
def ensure(name, script, *args, stale=lambda df: False):
    if not (OUT / name).exists() or stale(load(name)):
        subprocess.run([sys.executable, str(ANALYSIS / script), *args], check=True)
    return load(name)
avail = ensure('availability.csv', 'availability.py', '--max-f', '300', '--max-h', '300',
               stale=lambda d: 'pair' not in d or not (OUT / 'trade_waits.parquet').exists())
waits = load('trade_waits.parquet')
cents = ensure('gap_cents.csv', 'gap_cents.py', '--freshness', '5', '10', '60', stale=lambda d: 'role' not in d)
sweep = ensure('screen_sweep.csv', 'screen_sweep.py', '--max-f', '300')
m = gaps.read_export(OUT / 'matches.parquet')   # ticks.matches() plus cohort and complete_market_set
m['league'] = m.league.astype(str)
m['kickoff_iso'] = pd.to_datetime(m.kickoff_iso.astype(str), utc=True)
m['combo'] = m.league + ' ' + m.season.astype(str) + '/' + (m.season % 100 + 1).astype(str).str.zfill(2)
COMBO = m.set_index('match_id').combo
for df in (avail, cents, waits, sweep):
    df['combo'] = df.league + ' ' + df.season.astype(str) + '/' + (df.season % 100 + 1).astype(str).str.zfill(2)
avail_all = avail[avail.pair == 'all/all']   # both taker sides pooled, the pipeline's view
FIXTURES = m.groupby('combo').size()
def export(name, columns=None):
    df = gaps.read_export(OUT / f'{name}.parquet', columns)
    df['combo'] = df.match_id.map(COMBO)
    return df
audit = export('audit')
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': .3, 'axes.titlesize': 9, 'font.size': 9})
from competition import BLUE, RED, GREEN, PURPLE, grid, finish, bars, phase_coverage, select_candidates
print(f'{len(m):,} fixtures, {m.combo.nunique()} competition-seasons; phases {PHASES}')

## 1. Population by competition and season

Same universe as 02 (linked on both venues, grounded to API-Football), broken out by `season` (API-Football season start year; 2025 = 2025/26). Trades per fixture is the number to keep in mind for everything below: the World Cup is 18× the Premier League and 66× Ligue 1.

**1X2 within each venue's soccer tape.** The second table is the raw soccer pull under `research/data/raw/soccer` (every market type, every series / condition, to about August 2026; [`market_type_share.py`](../market_type_share.py)): files, bytes, trade rows and contracts of the 1X2 type (Kalshi `game` series, Polymarket `moneyline`, which carries the draw) as a share of that venue's total. Trade rows are raw fills, not the per-second prints used below; contracts are Kalshi `count` and Polymarket `size`. This is the universe the 1X2 study is cut from, not the linked fixture population above.

In [ ]:
cov = m.merge(audit[['match_id', 'rows', 'pm_rows', 'k_rows', 'window_rows']], on='match_id')
cov['both_traded'] = (cov.pm_rows > 0) & (cov.k_rows > 0)
pop = cov.groupby(['combo'], observed=True).agg(fixtures=('match_id', 'size'), both_traded=('both_traded', 'sum'),
                                              complete_1x2=('complete_market_set', 'sum'), trades=('rows', 'sum'), pm_trades=('pm_rows', 'sum'), k_trades=('k_rows', 'sum'),
                                              first_kickoff=('kickoff_iso', 'min'), last_kickoff=('kickoff_iso', 'max'))
for c in ('trades', 'pm_trades', 'k_trades'):
    pop[f'{c}_per_fixture'] = (pop[c] / pop.fixtures).round(0).astype(int)
pop = pop.sort_values('trades_per_fixture', ascending=False)
display(pop)
fig, ax = plt.subplots(figsize=(12, 3.6))
x = np.arange(len(pop))
ax.bar(x - .2, pop.pm_trades_per_fixture, .4, color=BLUE, label='Polymarket'); ax.bar(x + .2, pop.k_trades_per_fixture, .4, color=RED, label='Kalshi')
ax.set_yscale('log'); ax.set_xticks(x); ax.set_xticklabels(pop.index, rotation=30, ha='right', fontsize=8); ax.set_ylabel('trades per fixture (log)')
ax.legend(); ax.set_title('Trades per fixture by competition-season'); plt.tight_layout(); plt.show()

share = ensure('market_type_share.csv', 'market_type_share.py')
one_x_two = share[share.market_type.isin(['game', 'moneyline'])].set_index('venue')
print('1X2 share of each venue\'s raw soccer tape (%):')
display(one_x_two[['market_type', 'files', 'bytes_pct', 'trades', 'trades_pct', 'contracts', 'contracts_pct']].round(1))
print('Largest market types by trade rows:')
display(share.sort_values(['venue', 'trades'], ascending=[True, False]).groupby('venue').head(6)
        .set_index(['venue', 'market_type'])[['files', 'bytes_pct', 'trades_pct', 'contracts_pct']].round(1))

## 2. Availability and directional trade waiting — full population

Freshness F looks backward: a venue is fresh when its latest print is at most F seconds old. Forward horizon H asks whether the other venue prints strictly after the trigger and within H seconds. These answer different questions and use different denominators.

Availability uses the entire phase clock for all three outcomes in every fixture, including silent outcomes. Duration intervals are split at phase boundaries. The in-play split at 55 minutes is a clock-based diagnostic, not an observed halftime boundary. The full-population curves and state split (§2a) run before candidate selection; the per-action-pair split (§2c) and the trade waiting (§2d) are then confined to the candidate cells. Low dual freshness does not by itself rule out an asynchronous strategy.

In [ ]:
PH5 = ['pre_24h_to_1h', 'pre_last_hour', 'post_0_to_55m', 'post_55_to_105m', 'post_105_to_135m']
colors5 = dict(zip(PH5, [BLUE, RED, GREEN, '#19a974', PURPLE]))
combos = FIXTURES.sort_index()
fig, axes = grid(2, 5, w=4.2, h=3.6, sharex=True, sharey=True)
for ax, (combo, n) in zip(axes.flat, combos.items()):
    d = avail_all[avail_all.combo == combo]
    for ph in PH5:
        q = d[d.phase == ph].sort_values('freshness')
        ax.plot(q.freshness, q.fresh_pct_of_clock, color=colors5[ph], lw=1.8, label=ph)
    ax.set_title(f'{combo}  (n={n})'); ax.set_ylim(0, 100)
for ax in axes[-1]: ax.set_xlabel('Freshness F (s)')
for ax in axes[:, 0]: ax.set_ylabel('% of outcome-seconds with both venues fresh')
axes.flat[0].legend(fontsize=8, loc='upper left')
fig.suptitle('How much of the clock has a fresh cross-venue pair?', fontsize=13); fig.tight_layout(); plt.show()

# merge the two in-play halves (weighted by length) into the pipeline's post_0_to_105m
w = {'post_0_to_55m': 55 * 60, 'post_55_to_105m': 50 * 60}
cov4 = phase_coverage(avail_all)
h60 = avail_all[(avail_all.freshness == 60) & avail_all.phase.isin(w)].pivot(index='combo', columns='phase', values='fresh_pct_of_clock')
print(f'largest |second - first half| coverage difference at F=60: {(h60.post_55_to_105m - h60.post_0_to_55m).abs().max():.1f} pp -> keep post_0_to_105m merged')



### 2a. Where does the rest of the clock go?

Four mutually exclusive states sum to 100%: both fresh, only Kalshi fresh, only PM fresh, neither fresh. “Neither fresh” means neither venue traded in the preceding F seconds; it does not mean the data are missing. Conversely, a second without a new trade can still have both venues fresh. A missing tape file stops computation rather than being classified as inactivity.

Denominator: fixture × outcome × phase seconds. The plots retain all competition-seasons, at common F = 5 / 10 / 60 seconds, before applying the candidate rule. Prints before the analysis start are unavailable, so initial freshness and “no prior print” refer to the observed analysis window.

In [ ]:
STATE_COLS = ['fresh_pct_of_clock', 'k_only_pct_of_clock', 'pm_only_pct_of_clock', 'neither_pct_of_clock']
STATE_NAMES = ['Both fresh', 'Only Kalshi fresh', 'Only PM fresh', 'Neither fresh']
STATE_COLORS = [GREEN, RED, BLUE, '#bdbdbd']
def pool_states(a):
    """Four exclusive clock states per combo × pair × phase × F, the two in-play halves pooled by length."""
    half = a[a.phase.isin(w)].assign(wt=lambda d: d.phase.map(w))
    merged = (half.assign(**{col: half[col] * half.wt for col in STATE_COLS})
        .groupby(['combo', 'pair', 'freshness'])[STATE_COLS + ['wt']].sum())
    merged[STATE_COLS] = merged[STATE_COLS].div(merged.wt, axis=0)
    keep = ['combo', 'pair', 'phase', 'freshness'] + STATE_COLS
    out = pd.concat([a[~a.phase.isin(w)][keep], merged.reset_index().assign(phase='post_0_to_105m')[keep]], ignore_index=True)
    assert np.allclose(out[STATE_COLS].sum(axis=1), 100, atol=1e-6)
    return out
state4 = pool_states(avail)
for F in (5, 10, 60):
    fig, axes = grid(2, 2, w=9, h=4.7, sharex=True)
    for ax, ph in zip(axes.flat, PHASES):
        d = state4[(state4.pair == 'all/all') & (state4.phase == ph) & (state4.freshness == F)].set_index('combo').reindex(combos.index)
        left = np.zeros(len(d))
        for col, label, color in zip(STATE_COLS, STATE_NAMES, STATE_COLORS):
            ax.barh(d.index, d[col], left=left, color=color, label=label)
            left += d[col].to_numpy()
        ax.set_title(ph); ax.set_xlim(0, 100); ax.set_xlabel('% of all outcome-seconds'); ax.invert_yaxis()
    axes.flat[0].legend(fontsize=7, loc='lower right')
    fig.suptitle(f'Full-clock activity states: F={F}s'); fig.tight_layout(); plt.show()

### 2b. Candidate cells for the fresh-pair gap study

A competition-season × phase qualifies if dual freshness at F=60s covers at least 70% of its clock (rounded to a whole percent). Choose the smallest F in 5 / 10 / 60s reaching that threshold. This is a scope rule for the following fresh-pair price comparisons and for the finer activity diagnostics in §2c–d, not a rejection of asynchronous opportunities elsewhere. The full-population diagnostics above remain independent of this selection.

**Fixture concentration.** Every rate below pools observations across a cell's fixtures, so a few busy matches could dominate it. For each candidate cell the last table and the Lorenz curves count the fresh maker/maker observations (both directions, all outcomes, at the cell's F; the §5–8 population) per fixture: the share held by the single busiest fixture, the top 5 and the top 10% of fixtures, the Gini coefficient, and the median fixture. The uncertainty of any pooled rate is therefore closer to the fixture count than to the observation count.

In [ ]:
SCREEN_GRID = sorted(set(gaps.read_export(OUT / 'screen.parquet', ['freshness']).freshness.astype(int)))
table, CANDS, F_PIPE = select_candidates(cov4, FIXTURES, PHASES, SCREEN_GRID)
LABEL = {c: f'{c[0]}' + chr(10) + f'{c[1]}  F={c[2]}s' for c in CANDS}
CELLS = [LABEL[c].replace(chr(10), ' | ') for c in CANDS]
def by_cell(frame, extra=lambda c: {}):
    """Rows of the candidate cells (combo × phase), labelled."""
    return pd.concat([frame[(frame.combo == c[0]) & (frame.phase == c[1])].assign(cell=cl, **extra(c)) for c, cl in zip(CANDS, CELLS)], ignore_index=True)
display(table)
print('pipeline F used:', {LABEL[c].replace(chr(10), ' | '): F_PIPE[c] for c in CANDS})

# Concentration: do a few fixtures supply most of a cell's observations?
obs = export('screen', ['route', 'direction', 'outcome', 'phase', 'freshness', 'n', 'match_id'])
obs = by_cell(obs[obs.route == 'maker/maker'], lambda c: {'F_cell': c[2]})
obs = obs[obs.freshness == obs.F_cell].groupby(['cell', 'match_id']).n.sum()
conc = []
fig, ax = plt.subplots(figsize=(5.5, 4.6))
for cl, color in zip(CELLS, [BLUE, RED, GREEN, PURPLE]):
    v = np.sort(obs.loc[cl].to_numpy())[::-1]
    k, share_ = len(v), np.cumsum(v) / v.sum()
    lorenz = np.cumsum(v[::-1]) / v.sum()   # ascending Lorenz curve for the Gini
    conc.append(dict(cell=cl, fixtures=k, observations=int(v.sum()), top1_pct=100 * share_[0], top5_pct=100 * share_[min(4, k - 1)],
                     top10pct_fixtures_pct=100 * share_[max(int(np.ceil(k / 10)) - 1, 0)], gini=(k + 1 - 2 * lorenz.sum()) / k,
                     median_obs=float(np.median(v)), max_over_median=v[0] / np.median(v)))
    ax.plot(100 * np.arange(1, k + 1) / k, 100 * share_, color=color, lw=1.6, label=cl)
ax.plot([0, 100], [0, 100], color='grey', lw=.8, ls=':')
ax.set(xlabel='% of fixtures, busiest first', ylabel='% of maker/maker observations', xlim=(0, 100), ylim=(0, 100),
       title='Observation concentration across fixtures per candidate cell')
ax.legend(fontsize=7); plt.tight_layout(); plt.show()
conc = pd.DataFrame(conc).set_index('cell')
display(conc.round(2))

### 2c. Which leg goes stale? Activity states by taker-action pair, candidate cells

§2a pools both taker sides of each venue. The routes in §4 read one side per venue — for example `PM buy / K sell` is the pair a two-taker `PM YES + K NO` route prices from — so here each venue leg is a single taker side: a leg is fresh when that venue's latest print **on that side** is at most F old. The four exclusive states are the same as in §2a; `pooled` repeats the §2a row for reference. Bars use each candidate's F; the table shows F = 5 / 10 / 60s.

Denominator: fixture × outcome × phase seconds, as in §2a. A side that never printed within the observed window counts as never fresh.

In [ ]:
PURE = ['buy/sell', 'sell/buy', 'buy/buy', 'sell/sell']
PAIR_LABEL = {'all/all': 'pooled', 'buy/sell': 'PM buy / K sell', 'sell/buy': 'PM sell / K buy', 'buy/buy': 'PM buy / K buy', 'sell/sell': 'PM sell / K sell'}
fig, axes = grid(1, len(CANDS), w=4.6, h=3.0, sharex=True)
for ax, c in zip(axes[0], CANDS):
    d = state4[(state4.combo == c[0]) & (state4.phase == c[1]) & (state4.freshness == c[2])].set_index('pair').reindex(['all/all'] + PURE)
    left = np.zeros(len(d))
    for col, label, color in zip(STATE_COLS, STATE_NAMES, STATE_COLORS):
        ax.barh([PAIR_LABEL[p] for p in d.index], d[col], left=left, color=color, label=label)
        left += d[col].to_numpy()
    ax.set_xlim(0, 100); ax.invert_yaxis(); ax.set_xlabel('% of all outcome-seconds'); ax.set_title(f'F={c[2]}s', fontsize=8)
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc='lower center', ncol=4, fontsize=8, bbox_to_anchor=(0.5, -0.08))
finish(fig, axes, [''], [LABEL[c] for c in CANDS], 'PM leg / Kalshi leg', "Activity states by taker-action pair at each candidate's F")
legs = by_cell(state4[state4.freshness.isin([5, 10, 60])])
with pd.option_context('display.max_rows', None):
    display(legs.pivot_table(index=['cell', 'pair'], columns='freshness', values=['fresh_pct_of_clock', 'k_only_pct_of_clock', 'pm_only_pct_of_clock'])
            .reindex(['all/all'] + PURE, level=1).round(1))

### 2d. After a trade, how soon does the other venue trade? Candidate cells

For each fixture and outcome, each source-venue trade second counts once, merging buy/sell fills. The denominator includes **all** source trade seconds, without requiring an existing fresh pair. The numerator is a destination print in **(t, t+H]**. Same-second destination prints are reported separately and do not count as a forward response; those source triggers remain in the denominator.

H runs over every second from 1 to 60 on a linear axis (the export carries H up to 300 s; the curves are flat well before 60 s). Each trigger must have its entire H-second future inside its phase and the recorded analysis window; boundary-censored triggers are reported and excluded from the rate. The two in-play subphases are preserved for this check, even when pooled into the 0–105 minute reporting phase. Thus denominators vary with H and the displayed curves need not be strictly monotone. Only the candidate cells are shown.

The table also splits triggers by destination state at t: fresh (age ≤ F), stale (a previous observed print exists, age > F), or no prior observed print, at each cell's F and H = 60s. F and H are independent. Rates are pooled over source trade seconds, not averaged equally across fixtures. Zero eligible triggers give an undefined rate, not zero.

These are descriptive waiting probabilities, not evidence that the source causes or predicts destination activity above its baseline. That requires controls for market, time and activity, and uncertainty clustered by fixture.

In [ ]:
wait4 = waits.copy()
wait4['phase'] = wait4.phase.replace({'post_0_to_55m': 'post_0_to_105m', 'post_55_to_105m': 'post_0_to_105m'})
WAIT_COUNTS = ['triggers', 'same_second', 'eligible', 'censored', 'followed']
wg = wait4.groupby(['combo', 'phase', 'source', 'freshness', 'other_state', 'horizon'], observed=True)[WAIT_COUNTS].sum().reset_index()
wg['follow_pct'] = 100 * wg.followed / wg.eligible.replace(0, np.nan)
assert (wg.followed <= wg.eligible).all()
assert (wg.triggers == wg.eligible + wg.censored).all()
wg = by_cell(wg, lambda c: {'F_cell': c[2]})
SOURCES = [('KALSHI', RED, 'Kalshi → PM'), ('POLYMARKET', BLUE, 'PM → Kalshi')]
H_MAX = 60
# The all-trigger population is independent of F; use F=60 once.
fig, axes = grid(1, len(CANDS), w=4.6, h=3.2, sharey=True)
for ax, cl in zip(axes[0], CELLS):
    for src, color, label in SOURCES:
        d = wg[(wg.cell == cl) & (wg.source == src) & (wg.other_state == 'all') & (wg.freshness == 60) & (wg.horizon <= H_MAX)].sort_values('horizon')
        ax.plot(d.horizon, d.follow_pct, color=color, lw=1.5, marker='o', ms=2, label=label)
    ax.set_xlim(0, H_MAX); ax.set_ylim(0, 100); ax.set_xlabel('Forward horizon H (s)')
axes[0, 0].legend(fontsize=7, loc='lower right')
finish(fig, axes, [''], [LABEL[c] for c in CANDS], '% with a strictly later destination trade', f'All source trade seconds: H = 1 … {H_MAX} s')
print('Selected horizons, all triggers:')
display(wg[(wg.other_state == 'all') & (wg.freshness == 60) & wg.horizon.isin([1, 2, 3, 5, 10, 20, 30, 45, 60])]
        .pivot(index=['cell', 'source'], columns='horizon', values='follow_pct').round(1))

DETAIL_H = 60
wd = wg[(wg.freshness == wg.F_cell) & (wg.horizon == DETAIL_H)].copy()
wd['same_second_pct'] = 100 * wd.same_second / wd.triggers.replace(0, np.nan)
wd['censored_pct'] = 100 * wd.censored / wd.triggers.replace(0, np.nan)
print(f"Destination state at trigger, at each cell's F; forward waiting horizon H={DETAIL_H}s")
display(wd.pivot(index=['cell', 'source'], columns='other_state', values='follow_pct').reindex(columns=['all', 'fresh', 'stale', 'none']).round(1))
print('All-trigger denominators and same-second overlap (before boundary censoring):')
display(wd[wd.other_state == 'all'].set_index(['cell', 'source'])[['triggers', 'eligible', 'followed', 'same_second_pct', 'censored_pct']].round(1))

**Reading the current export.** At F=60s the Premier League in-play clock is 78.3% both fresh, 15.7% only Kalshi fresh, 2.6% only PM fresh and 3.4% neither (Champions League 69.7 / 18.7 / 4.4 / 7.2). Per taker side the picture is one-sided in a different way: the pure pairs are fresh on only 45–69% of that clock, and the missing leg is each venue's **sell** side. `PM buy / K sell` loses 22.0 pp of the Premier League clock to "only PM fresh" — the Kalshi sell-side print is stale — while `PM sell / K buy` loses 32.9 pp to "only Kalshi fresh"; `buy/buy` is the best-covered pure pair (68.7%) and `sell/sell` the worst (44.9%). Kalshi's buy-heavy flow (§4) is the same fact seen from the price side. In the World Cup's final pregame hour at F=10s the asymmetry is extreme: `PM sell / K buy` is 58.0% "only Kalshi fresh". A two-taker route reads exactly one of these pure pairs, so its reference is fresh on far less of the clock than the pooled 70% rule suggests.

With every H from 1 to 60s on a linear axis, the waiting curves are hockey sticks: after a Kalshi trade second, PM prints again within 1 / 5 / 10 / 60s in 13.8 / 47.6 / 64.7 / 93.6% of eligible Premier League triggers (Champions League 16.9 / 53.6 / 68.9 / 93.5%), and above 97% by 120s (beyond the plotted 60 s); the World Cup in play is at 87.9% by 5s and 99.9% by 60s. The reverse direction is faster everywhere (PM → Kalshi within 5s: 69.4% Premier League, 99.0% World Cup in play). When the destination is already stale at the trigger (at the cell's F, H=60s), the Premier League rates fall to 61.8% (Kalshi → PM) and 70.1% (PM → Kalshi), the Champions League to 60.0% and 36.0%; in the World Cup cells the stale groups still exceed 98%. These are event-weighted rates with phase-boundary censoring (about 2% of triggers); they cannot be inferred from the clock-weighted availability percentages.

The fresh-only analysis therefore describes a materially different activity population from the full tape. This motivates retaining the stale-trigger groups for subsequent price-response research; it does not establish causal leadership or profitability.

## 3. The state, freshness, and the signed gap

As in 02 §2: one row per second in which either venue printed, carrying each leg's last print; an **observation** is a print on either leg while the other leg is ≤ F old (event weighting, the taker's view); the same print weighted by the seconds its pair then stays fresh, `clip(F − age, 0, dur)`, gives the time weighting (the resting maker's view). Bars below are the share of fresh prints by pooled PM − K in cents: the centre bucket (−1, 1) holds the sub-cent noise, `[1, 2)` / `(−2, −1]` hold the exact one-cent gaps both venues quote, and the ends are open at ±4 c; lighter bars are time-weighted. Means in the tables come from exact sums, not bucket midpoints. Columns: candidate cells; rows: outcome selection (`team_win` = home + away, priced separately, pooled only for reporting).

**Second split: matchup class × role.** The same distribution, with the fixture first classed as **balanced** or **unequal** by 05's shared label ([`fixture_labels.parquet`](../outputs/relative_strength/followup/), VWAP30, `fusion_fallback`, x = 0.30, joined on `match_id` only in `gap_cents.py`; the "interface for future joins" of [05](05_relative_strength.ipynb)). In a balanced fixture the rows are `team_win` (home + away, both role `balanced`) and `draw`; in an unequal fixture they are `strong`, `weak` and `draw`, where strong is the side the fused pregame prices favour, whichever end it plays from. Fixtures without a usable label (`unknown`) are dropped from this split; a coverage table lists them.

This split is shown for the **in-play cells only**. The label is the fused VWAP of the last 30 pregame minutes, i.e. it is built from prints inside `pre_last_hour`: in that phase the class would be conditioned on the very prints whose gaps are measured, so the World Cup last-hour cell is left out. In play the label is fixed before the first observation.

In [ ]:
OUTCOME_SEL = {'all': ['home', 'draw', 'away'], 'team_win': ['home', 'away'], 'draw': ['draw']}
# matchup class (05's shared fixture label) x the outcome's role in it; `role` is 'draw' for the draw outcome
ROLE_SEL = {'balanced | team_win': ('balanced', ['balanced']), 'balanced | draw': ('balanced', ['draw']),
            'unequal | strong': ('unequal', ['strong']), 'unequal | weak': ('unequal', ['weak']), 'unequal | draw': ('unequal', ['draw'])}
POST_CANDS = [c for c in CANDS if c[1].startswith('post_')]   # the label uses the last pregame 30 min: only in-play cells are out of sample
C = list(gaps.GAP_CENTS)
cent_labels = {-4: '≤−4', -3: '(−4,−3]', -2: '(−3,−2]', -1: '(−2,−1]', 0: '(−1,1)', 1: '[1,2)', 2: '[2,3)', 3: '[3,4)', 4: '≥4'}
cent_labels = [cent_labels[c] for c in C]
def cell(frame, c, fcol='freshness'):
    combo, phase, F = c
    return frame[(frame.combo == combo) & (frame.phase == phase) & (frame[fcol] == F)]
def pick(frame, sel):
    """Rows of one selection: an outcome list, or a (matchup, roles) tuple."""
    if isinstance(sel, tuple):
        matchup, roles = sel
        return frame[(frame.matchup == matchup) & frame.role.isin(roles)]
    return frame[frame.outcome.isin(sel)]
def cent_stats(frame):
    g = frame.groupby('cents')[['n', 'time', 'gap_sum', 'abs_sum', 'gap_time_sum']].sum().reindex(C, fill_value=0)
    n, c = g.n.sum(), np.array(C)
    pct = 100 * g[['n', 'time']] / g[['n', 'time']].sum().replace(0, np.nan)
    return pct, dict(prints=int(n), mean_gap_c=g.gap_sum.sum() / max(n, 1), mean_abs_gap_c=g.abs_sum.sum() / max(n, 1),
                     ge_2c_pct=100 * g.n[np.abs(c) >= 2].sum() / max(n, 1),
                     pm_lower_1c_pct=100 * g.n[c <= -1].sum() / max(n, 1), pm_higher_1c_pct=100 * g.n[c >= 1].sum() / max(n, 1),
                     time_mean_gap_c=g.gap_time_sum.sum() / max(g.time.sum(), 1e-9))
def gap_distribution(selections, cands, title):
    """Signed-gap bars (rows: selections, columns: cells) and the per-cell summary rows."""
    rows = []
    fig, axes = grid(len(selections), len(cands), sharex=True, sharey=True)
    for i, (name, sel) in enumerate(selections.items()):
        for j, c in enumerate(cands):
            d = pick(cell(cents, c), sel); d = d[d.pair == 'all/all']
            pct, st = cent_stats(d)
            bars(axes[i, j], cent_labels, {'per print': pct.n, 'per second': pct.time}, [BLUE, '#b3b8ff'])
            axes[i, j].set_title(f'n={st["prints"]:,}', fontsize=8)
            rows.append(dict(outcome=name, cell=LABEL[c].replace(chr(10), ' | '), **st))
    for ax in axes[-1]: ax.set_xlabel('PM − K (cents)'); ax.tick_params(axis='x', rotation=45)
    axes[0, 0].legend(fontsize=8)
    finish(fig, axes, list(selections), [LABEL[c] for c in cands], '% of fresh prints', title)
    return pd.DataFrame(rows)
SUMMARY_COLS = ['mean_gap_c', 'mean_abs_gap_c', 'ge_2c_pct', 'pm_lower_1c_pct', 'pm_higher_1c_pct']
summary = gap_distribution(OUTCOME_SEL, CANDS, 'Signed gap distribution per candidate cell')
display(summary.pivot_table(index='cell', columns='outcome', values=SUMMARY_COLS).reindex(columns=list(OUTCOME_SEL), level=1).round(2))

# the matchup x role split partitions the same prints: check it, then show how many fall to unknown labels
parts = pd.concat([pick(cell(cents, c), ('unknown', ['unknown', 'draw'])).assign(cell=LABEL[c].replace(chr(10), ' | ')) for c in POST_CANDS]
                  + [pick(cell(cents, c), sel).assign(cell=LABEL[c].replace(chr(10), ' | ')) for c in POST_CANDS for sel in ROLE_SEL.values()])
assert np.isclose(parts[parts.pair == 'all/all'].n.sum(), sum(cent_stats(cell(cents, c)[cell(cents, c).pair == 'all/all'])[1]['prints'] for c in POST_CANDS))
role_summary = gap_distribution(ROLE_SEL, POST_CANDS, 'Signed gap distribution by matchup class and team role (in-play cells; 05 label, x = 0.30)')
display(role_summary.pivot_table(index='cell', columns='outcome', values=SUMMARY_COLS).reindex(columns=list(ROLE_SEL), level=1).round(2))
print('Fresh prints by matchup class (pooled pair, all outcomes); unknown = no usable pregame label:')
display(parts[parts.pair == 'all/all'].groupby(['cell', 'matchup']).n.sum().unstack(fill_value=0).astype(int).reindex(columns=['balanced', 'unequal', 'unknown']))

## 4. Action pairs and outcomes: the signed gap is a route's gross edge

Signed mean PM − K per fresh print for each of the nine taker-action pairs (within a heatmap: rows PM action, columns Kalshi action; `all` pools both sides), plus the two tails — the share of prints with PM − K ≤ −2 c and ≥ +2 c — and the print count. Heatmap rows are the outcome selections of §3 (`all`, `team_win` = home + away pooled, `draw`); the `all/all` square of each is the pooled gap of §3, so a systematically negative `draw` row says which venue prints the draw lower and its tails say how often the disagreement is large enough to matter. Home and away are not separated: the cross-venue mechanics are symmetric between them and their pooled gaps agree to within 0.2 c in every cell. The sign is not decoration: for the four pure pairs the signed gap **is** the pre-fee gross edge of one route in each direction (`1 − p_YES − p_NO` with the legs the route reads):

| pair (PM / K) | reads | PM − K < 0 means | PM − K > 0 means |
|---|---|---|---|
| buy / sell | PM ask-side print vs K bid-side print | taker/taker, PM YES + K NO (edge = −gap) | ordinary spread crossing, no route |
| sell / buy | PM bid-side vs K ask-side | ordinary spread crossing, no route | taker/taker, K YES + PM NO (edge = +gap) |
| buy / buy | both ask-side | taker/maker, PM YES + K NO (−gap) | maker/taker, K YES + PM NO (+gap) |
| sell / sell | both bid-side | maker/taker, PM YES + K NO (−gap) | taker/maker, K YES + PM NO (+gap) |

So a cell's left tail (PM − K ≤ −2 c, buckets `(−3,−2]` and below) is the share of prints on which the "PM YES" route of that pair had at least 2 c gross, the right tail (≥ +2 c) the same for "K YES". `|PM − K|` would merge the two and lose the direction; it stays useful only as a disagreement measure. Colour is the signed mean (blue = PM prints lower, red = higher); the pooled offset of about −0.4 c comes from Kalshi's buy-heavy flow printing at the ask.

The second heatmap repeats this for the matchup × role rows of §3 on the in-play cells: whether the pooled −0.4 c offset, and the draw row's sign, hold separately for the strong side, the weak side and the draw of unequal fixtures, and for the two rows of balanced ones. Same caveat as in §3: the class is a pregame label and is only out of sample in play.

In [ ]:
ORDER = ['buy', 'sell', 'all']
def gap_heatmap(selections, cands, cells, title):
    """3x3 action-pair heatmaps (rows: selections, columns: cells); returns the per-panel matrices and tidy rows."""
    fig, axes = grid(len(selections), len(cands), w=4.4, h=3.8)
    mats, rows = {}, []
    for i, (name, sel) in enumerate(selections.items()):
        for j, c in enumerate(cands):
            z = np.full((3, 3), np.nan); lo = np.zeros((3, 3)); hi = np.zeros((3, 3)); nn = np.zeros((3, 3))
            for (a, b) in product(range(3), repeat=2):
                d = pick(cell(cents, c), sel); d = d[d.pair == f'{ORDER[a]}/{ORDER[b]}']
                if len(d):
                    pct, st = cent_stats(d)
                    z[a, b], nn[a, b] = st['mean_gap_c'], st['prints']
                    lo[a, b], hi[a, b] = pct.n[np.array(C) <= -2].sum(), pct.n[np.array(C) >= 2].sum()
                    rows.append(dict(cell=cells[j], outcome=name, pm_action=ORDER[a], k_action=ORDER[b], mean_gap_c=z[a, b],
                                     le_m2c_pct=lo[a, b], ge_p2c_pct=hi[a, b], prints=int(nn[a, b])))
            mats[i, j] = (z, lo, hi, nn)
    vmax = max(np.nanmax(np.abs(z)) for z, *_ in mats.values())
    for (i, j), (z, lo, hi, nn) in mats.items():
        ax = axes[i, j]
        ax.imshow(z, cmap='RdBu_r', vmin=-vmax, vmax=vmax); ax.grid(False)
        for (a, b) in product(range(3), repeat=2):
            if np.isfinite(z[a, b]):
                ax.text(b, a, f'{z[a, b]:+.2f}c' + chr(10) + f'{lo[a, b]:.0f}% | {hi[a, b]:.0f}%' + chr(10) + f'{nn[a, b] / 1000:.0f}k', ha='center', va='center', fontsize=7)
        ax.set_xticks(range(3)); ax.set_xticklabels(ORDER, fontsize=8); ax.set_yticks(range(3)); ax.set_yticklabels(ORDER, fontsize=8)
        if i == len(selections) - 1: ax.set_xlabel('Kalshi taker action')
    finish(fig, axes, list(selections), [LABEL[c] for c in cands], 'PM taker action', title)
    return mats, pd.DataFrame(rows)
def pair_table(rows, selections):
    return (rows.pivot_table(index=['cell', 'outcome'], columns=['pm_action', 'k_action'], values='mean_gap_c')
            .reindex(list(selections), level=1).reindex(columns=ORDER, level=0).reindex(columns=ORDER, level=1).round(2))
mats, pair_rows = gap_heatmap(OUTCOME_SEL, CANDS, CELLS, 'By outcome and action pair: signed mean PM − K | % ≤ −2c | % ≥ +2c | prints (k)')
display(pair_table(pair_rows, OUTCOME_SEL))
POST_CELLS = [LABEL[c].replace(chr(10), ' | ') for c in POST_CANDS]
role_mats, role_pair_rows = gap_heatmap(ROLE_SEL, POST_CANDS, POST_CELLS, 'By matchup class, role and action pair (in-play cells): signed mean PM − K | % ≤ −2c | % ≥ +2c | prints (k)')
display(pair_table(role_pair_rows, ROLE_SEL))

## 5. Fee-adjusted last-print screens

At every route-leg print with the other leg ≤ F old, price both legs using their latest trades and subtract the fee model below. All four role combinations are included; route names list the PM role first and the Kalshi role second. Bars show the fraction of fresh observations with positive fee-adjusted reference edge, by route and direction.

**The fee model is not a flat rate.** Both venues charge a curve in the contract price: `rate × p × (1 − p)` per contract, largest at 50 c and zero at the extremes, where p is the price of the leg being bought (a NO leg pays on 1 − p; the curve is symmetric). Rates are `Config` inputs, checked against the venues' fee pages, not historical billing:

| leg | rate | rounding | at p = 50 c |
|---|---|---|---|
| Polymarket taker | 5% | none | 1.25 c / contract |
| Polymarket maker | 0 | — | 0 |
| Kalshi taker | 7% | fee rounded up to the cent on the 100-contract lot, plus the cent rounding of the lot's cash cost | 1.75 c / contract |
| Kalshi maker | 1.75% | same | 0.44 c / contract |

No rebates are assumed. The table below evaluates the curves at several prices; the freshness sweep at the end of the section uses the same routes and fees at every F from 1 to 300 s. These are last-print comparisons; future repricing diagnostics are in [02.2](02.2_cross_venue_confirmation_and_response.ipynb).

**Maker/maker: two passive-fill price references.** For `PM YES + K NO`, use the PM taker-sell YES print as the PM maker-buy YES reference, and one minus the Kalshi taker-buy YES print as the Kalshi maker-buy NO reference. The opposite direction reverses these actions. The reference net edge is `1 − PM cost − Kalshi cost − fees`.

A positive maker/maker reference edge assumes **both passive legs fill at those prices**. It does not measure joint fill probability, queue priority or the risk that only one leg fills. The route appears in this section's screen and freshness sweep. Because it is the only route whose positive share exceeds one half — the three routes with a taker leg are ≤ 0 after fees on most observations — §6 follows `maker/maker` by direction across F, and §7 and §8 study it alone; the three taker routes' event-time episodes are in [02.2 §5](02.2_cross_venue_confirmation_and_response.ipynb).

The last table repeats the maker/maker screen for the matchup × role split of §3 on the in-play cells (fixture label joined on `match_id`; `unknown` dropped): positive share and mean net edge per fresh observation, by direction. `K YES + PM NO` is Kalshi passive buy YES + Polymarket passive sell YES.

In [ ]:
fee_p = np.array([.05, .10, .30, .50, .70, .90, .95])
fee_table = pd.DataFrame({'price': fee_p,
    'PM taker': 100 * CFG.pm_rate * fee_p * (1 - fee_p),
    'PM maker': 0.0,
    'Kalshi taker': 100 * gaps.kalshi_fee(fee_p, CFG.lot, CFG.kalshi_taker_rate) / CFG.lot,
    'Kalshi maker': 100 * gaps.kalshi_fee(fee_p, CFG.lot, CFG.kalshi_maker_rate) / CFG.lot}).set_index('price')
print(f'Fee per contract in cents (lot = {CFG.lot}; PM {CFG.pm_rate:.0%} taker, Kalshi {CFG.kalshi_taker_rate:.0%} taker / {CFG.kalshi_maker_rate:.2%} maker):')
display(fee_table.round(3))

In [ ]:
scr = export('screen')
scr = scr[scr.combo.isin({c[0] for c in CANDS})]
ROUTES = ['taker/taker', 'taker/maker', 'maker/taker']
SCREEN_ROUTES = ROUTES + ['maker/maker']
DIRS = ['PM YES + K NO', 'K YES + PM NO']
def pipe_cell(frame, c):
    return frame[(frame.combo == c[0]) & (frame.phase == c[1]) & (frame.freshness == F_PIPE[c])]
def screen_rates(frame, by):
    g = frame.groupby(by, observed=True)[['n', 'n_lot', 'pos_0c', 'pos_0c_lot', 'pos_1c',
        'fresh_time', 'open_time_0c', 'edge_sum']].sum()
    g['fixtures'] = frame.groupby(by, observed=True).match_id.nunique()
    g['pos_pct'] = 100 * g.pos_0c / g.n.replace(0, np.nan)
    g['open_pct_of_fresh'] = 100 * g.open_time_0c / g.fresh_time.replace(0, np.nan)
    g['mean_net_c'] = 100 * g.edge_sum / g.n.replace(0, np.nan)
    return g.reset_index()
screen_cells = pd.concat([pipe_cell(scr, c).assign(cell=cl) for c, cl in zip(CANDS, CELLS)], ignore_index=True)
rates = []
fig, axes = grid(len(OUTCOME_SEL), len(CANDS), sharex=True, sharey=True)
for i, (name, outcomes) in enumerate(OUTCOME_SEL.items()):
    r = screen_rates(screen_cells[screen_cells.outcome.isin(outcomes)], ['cell', 'route', 'direction']).assign(outcome_sel=name)
    rates.append(r)
    for j, cl in enumerate(CELLS):
        d = r[r.cell == cl]
        piv = lambda col: {dr: d[d.direction == dr].set_index('route').reindex(SCREEN_ROUTES)[col].values for dr in DIRS}
        bars(axes[i, j], SCREEN_ROUTES, piv('pos_pct'), [BLUE, RED])
        axes[i, j].set_title(f'pipeline F={F_PIPE[CANDS[j]]}s   n={int(d.n.sum()):,}', fontsize=8)
axes[0, 0].legend(fontsize=8, title='positive vs last print', title_fontsize=7)
finish(fig, axes, list(OUTCOME_SEL), [LABEL[c] for c in CANDS], '% of observations', 'Positive after charged fees, by route and direction')
rates = pd.concat(rates, ignore_index=True)
route_summary = rates[rates.route.isin(['taker/taker', 'maker/maker'])]
for col in ['n', 'pos_pct', 'mean_net_c', 'open_pct_of_fresh']:
    print(f'Taker/taker and maker/maker reference screen: {col}')
    display(route_summary.pivot_table(index=['cell', 'route'], columns=['outcome_sel', 'direction'], values=col).reindex(columns=['all', 'team_win', 'draw'], level=0).round(2))
# Sections 6-8 study maker/maker only (episodes of the taker routes are in 02.2 §5).
FREQ_ROUTES = ['maker/maker']
scr_cells = screen_cells[screen_cells.route.isin(FREQ_ROUTES)].copy()

# maker/maker by matchup class x role (05's shared label), in-play cells only, as in section 3
LABELS = pd.read_parquet(ANALYSIS / 'outputs/relative_strength/followup/fixture_labels.parquet')
LABELS = LABELS[LABELS.definition.eq('vwap_30m') & LABELS.policy.eq('fusion_fallback') & LABELS.threshold.eq(0.30)]
mm_role = screen_cells[screen_cells.route.eq('maker/maker') & screen_cells.cell.isin(POST_CELLS)].merge(
    LABELS[['match_id', 'matchup', 'home_role', 'away_role']], on='match_id', how='left', validate='many_to_one')
mm_role['role'] = np.select([mm_role.outcome.eq('draw'), mm_role.outcome.eq('home')], ['draw', mm_role.home_role], mm_role.away_role)
mm_role['class'] = mm_role.matchup + ' | ' + mm_role.role.where(mm_role.role.ne('balanced'), 'team_win')
mm_role = screen_rates(mm_role[mm_role.matchup.ne('unknown')], ['cell', 'class', 'direction'])
print('Maker/maker reference screen by matchup class and role (in-play cells; 05 label, x = 0.30):')
display(mm_role.pivot_table(index=['cell', 'class'], columns='direction', values=['n', 'fixtures', 'pos_pct', 'mean_net_c'])
        .reindex(list(ROLE_SEL), level=1).reindex(columns=['n', 'fixtures', 'pos_pct', 'mean_net_c'], level=0).round(2))

In [ ]:
# Last-print positive rates at every freshness F = 1..300 s (screen_sweep.csv: the same route series and
# fee model as screen.parquet, which carries only the coarse pipeline grid). Markers every 5 s.
sw = by_cell(sweep).groupby(['cell', 'route', 'freshness'], observed=True)[['n', 'pos_0c']].sum().reset_index()
sw['pos_pct'] = 100 * sw.pos_0c / sw.n.replace(0, np.nan)
chk = screen_rates(pd.concat([scr[(scr.combo == c[0]) & (scr.phase == c[1])].assign(cell=cl) for c, cl in zip(CANDS, CELLS)]), ['cell', 'route', 'freshness'])
chk = chk.merge(sw, on=['cell', 'route', 'freshness'], suffixes=('', '_sweep'))
assert np.allclose(chk.n, chk.n_sweep) and np.allclose(chk.pos_0c, chk.pos_0c_sweep), 'screen_sweep.csv disagrees with screen.parquet'
fig, axes = grid(1, len(CANDS), w=4.6, h=3.2, sharey=True)
for j, c in enumerate(CANDS):
    for r_, color in zip(SCREEN_ROUTES, [BLUE, GREEN, RED, PURPLE]):
        d = sw[(sw.cell == CELLS[j]) & (sw.route == r_)].sort_values('freshness')
        axes[0, j].plot(d.freshness, d.pos_pct, color=color, lw=1.2, label=f'{r_} vs last print')
        q = d[d.freshness % 5 == 0]; axes[0, j].plot(q.freshness, q.pos_pct, 'o', ms=2.5, color=color)
    axes[0, j].axvline(c[2], color='grey', lw=.8, ls=':'); axes[0, j].set_xlim(0, 300); axes[0, j].set_xlabel('Freshness F (s)')
axes[0, 0].legend(fontsize=6)
finish(fig, axes, [''], [LABEL[c] for c in CANDS], '% of observations', "Last-print positive rates by freshness, F = 1 … 300 s (dotted line = the cell's F)")
display(sw[sw.freshness.isin([1, 5, 10, 15, 30, 60, 120, 300])].pivot(index=['cell', 'route'], columns='freshness', values='pos_pct').round(1))

## 6. Maker/maker by direction across freshness

The only route that is positive on more than half of its observations, followed across F = 1 … 60 s instead of at one cutoff. Because F is swept, the cells are competition-season × phase only; the candidate F of §2b plays no role here (in §3–5 the F in each column title *is* used — those statistics are evaluated at that cutoff).

Each direction of `maker/maker` reads one pure taker-action pair of §2c: `PM YES + K NO` prices PM's passive buy from the **PM sell** print and Kalshi's passive NO buy from the **K buy** print (`PM sell / K buy`); `K YES + PM NO` reads `PM buy / K sell`; `pooled` sums both directions and, for the clock states, uses the venue-level `all/all` pair. Six rows per cell, all with F on the x axis:

1. **clock states** of the pair the direction reads, one row per direction — the four exclusive states of §2a stacked vertically at F = 5, 10, … 60 s, as % of the outcome clock (§2c data);
2. **positive rate** — fee-adjusted maker/maker reference edge > 0, as % of fresh observations (§5 data);
3. **positive standing, % of clock** — the maker view in seconds: `open_time_0c` (seconds a fresh, positive maker/maker reference stands, `clip(F − age, 0, dur)` summed over positive observations) over the outcome clock. This is the seconds-space product P(both fresh) × P(positive | both fresh); the print-weighted rate of the row above is the taker view and the two need not agree. The table also lists the conditional version, `open_pct_of_fresh`;
4. **mean net edge after fees** in cents — `1 − PM cost − Kalshi cost − fees` under the §5 fee model (PM maker 0, Kalshi maker 1.75%), per fresh print (solid) and per fresh second (dashed, i.e. weighted by how long the pair then stays fresh).

Everything here is a pair of passive-fill *references*: a positive edge says the two venues' resting-side prints overlapped after fees, not that either leg would have filled.

In [ ]:
MM_DIRS = {'PM YES + K NO': 'sell/buy', 'K YES + PM NO': 'buy/sell', 'pooled': 'all/all'}   # direction -> PM action / K action it reads
MM_COLORS = {'PM YES + K NO': BLUE, 'K YES + PM NO': RED, 'pooled': '#555555'}
F_MAX_MM = 60
CELL_NOF = {cl: cl.split('  F=')[0] for cl in CELLS}
# clock states of the pair each direction reads (in-play halves pooled in state4)
st = by_cell(state4[state4.freshness <= F_MAX_MM]).rename(columns={'fresh_pct_of_clock': 'both_fresh_pct'})
st = pd.concat([st[st.pair == pair].assign(direction=d) for d, pair in MM_DIRS.items()], ignore_index=True)
# maker/maker sweep by direction, plus the pooled sum of both directions
mm = by_cell(sweep[(sweep.route == 'maker/maker') & (sweep.freshness <= F_MAX_MM)])
mm = pd.concat([mm, mm.assign(direction='pooled')], ignore_index=True)
mm = mm.groupby(['cell', 'direction', 'freshness'], observed=True)[['n', 'pos_0c', 'edge_sum', 'fresh_time', 'open_time_0c', 'edge_time_sum']].sum().reset_index()
mm['pos_pct'] = 100 * mm.pos_0c / mm.n.replace(0, np.nan)
mm['mean_net_c'] = 100 * mm.edge_sum / mm.n.replace(0, np.nan)
mm['time_mean_net_c'] = 100 * mm.edge_time_sum / mm.fresh_time.replace(0, np.nan)
mm = mm.merge(st[['cell', 'direction', 'freshness', 'both_fresh_pct', 'k_only_pct_of_clock', 'pm_only_pct_of_clock']], on=['cell', 'direction', 'freshness'])
# maker view, in seconds: share of the clock with a positive maker/maker reference standing (three outcome clocks per direction)
clock = {cl: 3 * FIXTURES[c[0]] * PHASE_SEC[c[1]] for c, cl in zip(CANDS, CELLS)}
mm['open_pct_of_clock'] = 100 * mm.open_time_0c / (mm.cell.map(clock) * np.where(mm.direction == 'pooled', 2, 1))
mm['open_pct_of_fresh'] = 100 * mm.open_time_0c / mm.fresh_time.replace(0, np.nan)
chk = scr[scr.route == 'maker/maker']; chk = by_cell(chk).groupby(['cell', 'direction', 'freshness'], observed=True).open_time_0c.sum().reset_index()
chk = chk.merge(mm, on=['cell', 'direction', 'freshness'], suffixes=('_screen', ''))
assert np.allclose(chk.open_time_0c_screen, chk.open_time_0c), 'screen_sweep.csv disagrees with screen.parquet'

F_BARS = list(range(5, F_MAX_MM + 1, 5))
ROWS = [f'{d}: clock states (%)' for d in MM_DIRS] + ['positive rate (% of fresh obs.)', 'positive standing: % of clock', 'mean net edge after fees (cents)']
fig, axes = grid(len(ROWS), len(CANDS), w=4.6, h=2.6, sharex=True, sharey='row')
for j, cl in enumerate(CELLS):
    for i, d in enumerate(MM_DIRS):   # stacked vertical bars, one row per direction, states as in §2a
        q = mm[(mm.cell == cl) & (mm.direction == d) & mm.freshness.isin(F_BARS)].sort_values('freshness')
        bottom = np.zeros(len(q))
        for col, name, color in zip(['both_fresh_pct', 'k_only_pct_of_clock', 'pm_only_pct_of_clock'], STATE_NAMES[:3], STATE_COLORS[:3]):
            axes[i, j].bar(q.freshness, q[col], bottom=bottom, width=3.6, color=color, label=name); bottom += q[col].to_numpy()
        axes[i, j].bar(q.freshness, 100 - bottom, bottom=bottom, width=3.6, color=STATE_COLORS[3], label=STATE_NAMES[3])
        axes[i, j].set_ylim(0, 100)
    for d, color in MM_COLORS.items():
        q = mm[(mm.cell == cl) & (mm.direction == d)].sort_values('freshness')
        axes[3, j].plot(q.freshness, q.pos_pct, color=color, label=d)
        axes[4, j].plot(q.freshness, q.open_pct_of_clock, color=color, label=d)
        axes[5, j].plot(q.freshness, q.mean_net_c, color=color, label=f'{d}: per print')
        axes[5, j].plot(q.freshness, q.time_mean_net_c, color=color, ls='--', lw=1, label='per fresh second' if d == 'pooled' else None)
    axes[5, j].axhline(0, color='k', lw=.6); axes[5, j].set_xlabel('Freshness F (s)'); axes[0, j].set_xlim(0, F_MAX_MM + 3)
    axes[3, j].set_ylim(0, 100); axes[4, j].set_ylim(0, 100)
axes[0, 0].legend(fontsize=6, ncol=2, loc='upper left')
for i in (3, 4, 5): axes[i, 0].legend(fontsize=6)
finish(fig, axes, ROWS, [CELL_NOF[cl] for cl in CELLS], '', 'Maker/maker by direction across freshness (competition × phase, F not fixed)')
tab = mm[mm.freshness.isin(range(5, F_MAX_MM + 1, 5))]
for col in ['both_fresh_pct', 'pos_pct', 'open_pct_of_fresh', 'open_pct_of_clock', 'mean_net_c', 'time_mean_net_c']:
    print(f'--- {col}')
    display(tab.pivot(index=['cell', 'direction'], columns='freshness', values=col).round(2))


## 7. Positive reference edge by YES price level

This conditional comparison uses each candidate cell's selected F, shown in its subplot subtitle. The three routes with a taker leg are dropped here: in §5 they are ≤ 0 after fees on well over half of their observations in every cell, so conditioning them would only split a small minority. Outcomes are pooled within each bucket, while the two route directions are shown separately and together as the original pooled result. Each rate is positive last-print observations divided by all observations **in that same bucket**; it is not the share of all positives contributed by that bucket. Tables retain the observation counts so small groups remain visible.

`PM YES + K NO` reads PM sell-YES and Kalshi buy-YES prints; `PM NO + K YES` reads PM buy-YES and Kalshi sell-YES prints. Maker/maker uses two passive-fill price references, as in §5, and does not measure joint fill probability: a bucket with a high positive share is a bucket where the two venues' resting-side references overlap, not a fill rate.

In [ ]:
cd = export('conditions')
if 'freshness' not in cd:
    raise ValueError('conditions.parquet predates the freshness sweep; rerun gaps.run')
cd = by_cell(cd, lambda c: {'F_cell': F_PIPE[c]})
cd = cd[(cd.route == 'maker/maker') & (cd.freshness == cd.F_cell)]
CONDITION_DIRECTIONS = {
    'PM YES + K NO': 'PM YES + K NO',
    'K YES + PM NO': 'PM NO + K YES',
}
CONDITION_COLORS = [BLUE, RED, '#555555']

def plot_price_conditions():
    keys = ['cell', 'direction', 'price_bucket']
    directed = cd.groupby(keys, observed=True)[['n', 'pos_0c']].sum().reset_index()
    directed['direction'] = directed.direction.map(CONDITION_DIRECTIONS)
    pooled = (cd.groupby(['cell', 'price_bucket'], observed=True)[['n', 'pos_0c']].sum().reset_index()
              .assign(direction='Pooled'))
    g = pd.concat([directed, pooled], ignore_index=True)
    g['pos_pct'] = 100 * g.pos_0c / g.n.replace(0, np.nan)
    levels = list(gaps.PRICE_BUCKETS)
    fig, axes = grid(1, len(CANDS), w=4.6, h=3.2, sharey=True)
    for j, cl in enumerate(CELLS):
        d = g[g.cell == cl]
        series = {label: d[d.direction == label].set_index('price_bucket').reindex(levels).pos_pct.values
                  for label in CONDITION_DIRECTIONS.values()}
        series['Pooled'] = d[d.direction == 'Pooled'].set_index('price_bucket').reindex(levels).pos_pct.values
        bars(axes[0, j], levels, series, CONDITION_COLORS)
        observations = int(cd[cd.cell == cl].n.sum())
        axes[0, j].set_title(f'observations={observations:,}', fontsize=8)
        axes[0, j].tick_params(axis='x', rotation=45)
    axes[0, 0].legend(fontsize=7)
    finish(fig, axes, [''], [LABEL[c] for c in CANDS], '% positive within bucket (maker/maker)',
           'Maker/maker positive reference edge by YES price level — each candidate at its selected F')
    g['price_bucket'] = pd.Categorical(g.price_bucket, levels, ordered=True)
    wide = lambda col: g.pivot_table(index=['cell', 'direction'], columns='price_bucket', values=col, observed=True).reindex(columns=levels)
    print('Observations per bar (thousands):')
    display((wide('n') / 1000).round(1))
    print('Positive % per bar:')
    display(wide('pos_pct').round(1))


### Price level

Buckets refer to the YES acquisition-price reference on the route's YES-holding venue: PM for `PM YES + K NO`, and Kalshi for `PM NO + K YES`.

In [ ]:
plot_price_conditions()

## 8. How frequent? Per fixture-hour, per fixture, share of the clock

02 §7 per candidate cell at its F, for `maker/maker` only (the taker routes are ≤ 0 after fees on most observations, §5):

1. **per fixture-hour** — positive observations per hour of the cell's phase (phase length × fixtures as denominator);
2. **per fixture** — how many fixtures show at least one / 5+ / 20+ last-print positives, and how concentrated the total is;
3. **share of the clock** — time-weighted: seconds with a fresh, positive last-print edge over the seconds with any fresh pair, and over the whole clock (six clocks per fixture: three outcomes × two directions).

A maker/maker positive is a second in which the two venues' resting-side references overlap; these counts say how often and for how long that overlap stands on the tape, not how often both passive legs would fill.

In [ ]:
ph = screen_rates(scr_cells, ['cell', 'route'])
ph['combo'] = ph.cell.map({LABEL[c].replace(chr(10), ' | '): c[0] for c in CANDS}); ph['phase'] = ph.cell.map({LABEL[c].replace(chr(10), ' | '): c[1] for c in CANDS})
ph['fixture_hours'] = ph.combo.map(FIXTURES).astype(float) * ph.phase.map(PHASE_SEC) / 3600
ph['pos_per_fixture_hour'] = ph.pos_0c / ph.fixture_hours
ph['open_pct_of_clock'] = 100 * ph.open_time_0c / (6 * ph.combo.map(FIXTURES).astype(float) * ph.phase.map(PHASE_SEC))
fig, axes = grid(1, len(CANDS), w=4.4, h=3.2, sharey=True)
for j, cl in enumerate(CELLS):
    d = ph[ph.cell == cl].set_index('route').reindex(FREQ_ROUTES)
    bars(axes[0, j], FREQ_ROUTES, {'positive vs last print': d.pos_per_fixture_hour.values}, [PURPLE])
axes[0, 0].legend(fontsize=7)
finish(fig, axes, [''], [LABEL[c] for c in CANDS], 'per fixture-hour', 'Maker/maker positive observations per fixture-hour')
display(ph.set_index(['cell', 'route'])[['n', 'pos_per_fixture_hour', 'open_pct_of_fresh', 'open_pct_of_clock']].round(2))

per_fx = scr_cells.groupby(['cell', 'match_id', 'route'], observed=True).pos_0c.sum().reset_index()
dist = []
for c in CANDS:
    cl = LABEL[c].replace(chr(10), ' | '); ids = m[m.combo == c[0]].match_id
    for r_ in FREQ_ROUTES:
        s = per_fx[(per_fx.cell == cl) & (per_fx.route == r_)].set_index('match_id').pos_0c.reindex(ids, fill_value=0)
        dist.append(dict(cell=cl, route=r_, fixtures=len(s), with_any=int((s > 0).sum()), with_5_plus=int((s >= 5).sum()),
                         with_20_plus=int((s >= 20).sum()), median=float(s.median()), p90=float(s.quantile(.9)),
                         top10_share_pct=100 * s.nlargest(10).sum() / max(s.sum(), 1)))
display(pd.DataFrame(dist).set_index(['cell', 'route']).round(1))

## Summary

- Full-clock activity and source-trade waiting use different denominators. Keep the fresh, stale and no-prior-print trigger groups separate when interpreting §2. Per taker side (§2c) the scarce leg is each venue's sell side: the pure action pairs are fresh on only 45–69% of the club cells' clock against 70–78% pooled. The waiting curves (§2d) rise steeply within the first ten seconds and are flat before H = 60 s.
- The 70% rule selects World Cup in play (F=5s), World Cup's final pregame hour (F=10s), and Premier League / Champions League 2025/26 in play (F=60s). This scopes the fresh-pair gap screen; it does not rule out asynchronous strategies elsewhere.
- §3–7 describe last-print reference prices, fee-adjusted screens (one fee model, no rebates) and their frequency. Only `maker/maker` is positive on more than half of its observations, so §6 follows that route by direction across F and §7–8 condition and count it alone; its positives are overlapping resting-side references, not fills. Changes with F can reflect stale references; a positive screen is not an executable opportunity.
- The next stage is [02.2](02.2_cross_venue_confirmation_and_response.ipynb): event-time episode durations, subsequent prices, passive-fill diagnostics and baskets.